# Customer Segmentation & Churn Pattern Analytics in European Banking

## Notebook 02: Data Cleaning & Preparation

---

### Project Overview

After validating the dataset, the next step is to prepare it for analysis. Data cleaning improves the quality and usability of the dataset by removing non-informative features, preparing categorical variables for analysis, and creating meaningful customer segments.

The transformations performed in this notebook ensure that the dataset is structured, consistent, and ready for exploratory data analysis and customer segmentation.

---

## Objectives

The objectives of this notebook are to:

A. **Remove non-analytical fields** that do not contribute to customer churn analysis.

B. **Convert categorical variables** into a format suitable for grouping and analysis.

C. **Create derived segmentation fields** to support customer segmentation and business insights.

D. **Verifying and saving cleaned data**

---

## Expected Outcome

By the end of this notebook, the dataset will be cleaned, enriched with new segmentation features, and ready for exploratory data analysis.

---

## Load the Dataset

The validated dataset is loaded to begin the data cleaning and preparation process. All transformations performed in this notebook will be applied to a working copy of the dataset, preserving the original raw data.

In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../data/European_Bank.csv")

# Create a working copy
clean_df = df.copy()

clean_df.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


---

## A. Removing Non-Analytical Fields

Not every column in the dataset contributes meaningful information to customer churn analysis.

Columns that serve only as identifiers or contain no analytical value can be removed to simplify the dataset and improve the efficiency of subsequent analysis.

The following columns will be removed:

- **Surname** – Personal identifier that does not influence churn behaviour.
- **CustomerId** – Unique identifier used for record identification only.
- **Year** – Contains a single constant value for all records and provides no analytical value.

In [3]:
# Remove non-analytical columns

columns_to_remove = [
    "Surname",
    "CustomerId",
    "Year"
]

clean_df = clean_df.drop(columns=columns_to_remove)

clean_df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


---

## B. Converting Categorical Variables

The dataset contains categorical variables that describe customer characteristics rather than numerical measurements.

To improve memory efficiency and prepare these features for grouping and aggregation, categorical columns are converted to the **category** data type.

This transformation does not change the underlying values and preserves the original category names for easier interpretation during analysis.

In [4]:
# Convert categorical columns

categorical_columns = [
    "Geography",
    "Gender"
]

clean_df[categorical_columns] = clean_df[categorical_columns].astype("category")

clean_df[categorical_columns].dtypes

Geography    category
Gender       category
dtype: object

### Observation

The categorical variables were successfully converted to the **category** data type.

This optimization improves memory efficiency while preserving the original category labels, making the dataset more suitable for grouping, aggregation, and visualization in the upcoming analysis.

---

# C. Create Derived Segmentation Fields

Raw numerical values often provide limited business context when analyzed individually. Creating derived segmentation fields transforms continuous variables into meaningful customer groups, making it easier to identify patterns, compare customer segments, and communicate insights effectively.

The derived features created in this section will be used throughout the project for exploratory data analysis, customer segmentation, KPI calculation, and the interactive dashboard.

The following segmentation fields will be created:

- **Age Group**
- **Credit Score Band**
- **Balance Category**
- **Tenure Group**

#### 1. Creating Age Groups

Rather than analysing each individual age, customers are grouped into broader age categories. This makes it easier to compare customer behaviour across different life stages and identify age-related churn patterns.

The age groups used in this project are:

| Age Range | Age Group |
|-----------:|-----------|
| Below 30 | Below 30 |
| 30–45 | 30–45 |
| 46–60 | 46–60 |
| Above 60 | Above 60 |

In [10]:
# Create Age Group

age_bins = [0, 30, 45, 60, float("inf")]

age_labels = [
    "Below 30",
    "30–45",
    "46–60",
    "Above 60"
]

clean_df["AgeGroup"] = pd.cut(
    clean_df["Age"],
    bins=age_bins,
    labels=age_labels
)
clean_df[["Age", "AgeGroup"]].sample(5, random_state=42)

,Age,AgeGroup
6252,32,30–45
4684,43,30–45
1731,44,30–45
4742,59,46–60
4521,27,Below 30


In [6]:
age_group_distribution = (
    clean_df["AgeGroup"]
    .value_counts()
    .sort_index()
    .reset_index(name="Customer Count")
)

age_group_distribution["Percentage (%)"] = (
    age_group_distribution["Customer Count"] / len(clean_df) * 100
).round(2)

age_group_distribution

,AgeGroup,Customer Count,Percentage (%)
0,Below 30,1968,19.68
1,30–45,5921,59.21
2,46–60,1647,16.47
3,Above 60,464,4.64


### Observation

The **AgeGroup** feature was successfully created by grouping customers into four meaningful age segments.

The majority of customers (**59.21%**) fall within the **30–45** age group, making it the largest customer segment in the dataset. Customers **below 30** account for **19.68%**, while those aged **46–60** represent **16.47%**. Only **4.64%** of customers are **above 60**, making it the smallest segment.

These age groups provide a more meaningful basis for customer segmentation and will be used throughout the project to analyze demographic trends and churn behaviour.

#### 2. Creating Credit Score Bands

Credit scores indicate a customer's financial reliability and are an important factor in banking analytics. Rather than analysing individual scores, customers are grouped into broader credit score bands.

This segmentation simplifies comparisons between customer groups and helps identify whether creditworthiness is associated with customer churn.

The credit score bands used in this project are:

| Credit Score | Band |
|--------------:|------|
| Below 580 | Poor |
| 580–669 | Fair |
| 670–739 | Good |
| 740 and above | Excellent |

In [11]:
# Create Credit Score Bands

credit_bins = [0, 580, 670, 740, float("inf")]

credit_labels = [
    "Poor",
    "Fair",
    "Good",
    "Excellent"
]

clean_df["CreditScoreBand"] = pd.cut(
    clean_df["CreditScore"],
    bins=credit_bins,
    labels=credit_labels
)

clean_df[["CreditScore", "CreditScoreBand"]].sample(5, random_state=42)

,CreditScore,CreditScoreBand
6252,596,Fair
4684,623,Fair
1731,601,Fair
4742,506,Poor
4521,560,Poor


In [12]:
credit_band_distribution = (
    clean_df["CreditScoreBand"]
    .value_counts()
    .sort_index()
    .reset_index(name="Customer Count")
)

credit_band_distribution["Percentage (%)"] = (
    credit_band_distribution["Customer Count"] / len(clean_df) * 100
).round(2)

credit_band_distribution

,CreditScoreBand,Customer Count,Percentage (%)
0,Poor,2393,23.93
1,Fair,3350,33.50
2,Good,2397,23.97
3,Excellent,1860,18.60


### Observation

The **CreditScoreBand** feature was successfully created by classifying customers into four creditworthiness categories.

The largest segment consists of customers with **Fair** credit scores (**33.50%**), followed by **Good** (**23.97%**) and **Poor** (**23.93%**) credit scores. Customers with **Excellent** credit scores represent **18.60%** of the dataset, making it the smallest credit score segment.

This derived feature simplifies the analysis of customer credit profiles and will be used to investigate whether customers with different levels of creditworthiness exhibit different churn patterns.

#### 3. Creating Balance Categories

Customer account balances vary considerably across the dataset. To simplify analysis, balances are grouped into meaningful categories based on their value.

These balance categories provide a clearer understanding of customer value segments and will support comparisons of churn behaviour across different account balance levels.

The balance categories used in this project are:

| Balance | Category |
|---------:|----------|
| 0 | Zero Balance |
| 0–50,000 | Low Balance |
| 50,000–100,000 | Medium Balance |
| Above 100,000 | High Balance |

In [13]:
# Create Balance Categories

balance_conditions = [
    clean_df["Balance"] == 0,
    (clean_df["Balance"] > 0) & (clean_df["Balance"] <= 50000),
    (clean_df["Balance"] > 50000) & (clean_df["Balance"] <= 100000),
    clean_df["Balance"] > 100000
]

balance_labels = [
    "Zero Balance",
    "Low Balance",
    "Medium Balance",
    "High Balance"
]

clean_df["BalanceCategory"] = np.select(
    balance_conditions,
    balance_labels,
    default="Unknown"
)

clean_df[["Balance", "BalanceCategory"]].sample(5, random_state=42)

,Balance,BalanceCategory
6252,96709.07,Medium Balance
4684,0.00,Zero Balance
1731,0.00,Zero Balance
4742,119152.10,High Balance
4521,124995.98,High Balance


In [14]:
balance_category_distribution = (
    clean_df["BalanceCategory"]
    .value_counts()
    .rename_axis("Balance Category")
    .reset_index(name="Customer Count")
)

balance_category_distribution["Percentage (%)"] = (
    balance_category_distribution["Customer Count"] / len(clean_df) * 100
).round(2)

balance_category_distribution

,Balance Category,Customer Count,Percentage (%)
0,High Balance,4799,47.99
1,Zero Balance,3617,36.17
2,Medium Balance,1509,15.09
3,Low Balance,75,0.75


### Observation

The **BalanceCategory** feature was successfully created by grouping customers according to their account balance.

Nearly half of the customers (**47.99%**) belong to the **High Balance** category, while **36.17%** have a **Zero Balance** account. Customers with a **Medium Balance** represent **15.09%** of the dataset, whereas only **0.75%** fall into the **Low Balance** category.

These balance categories provide a clearer representation of customer value segments and will support further analysis of how account balances relate to customer churn and engagement.

#### 4. Creating Tenure Groups

Customer tenure represents the length of time a customer has maintained a relationship with the bank. Instead of analysing individual tenure values, customers are grouped into broader loyalty segments.

These tenure groups simplify comparisons between newer and long-term customers, making it easier to identify how customer loyalty relates to churn behaviour.

The tenure groups used in this project are:

| Tenure (Years) | Group |
|---------------:|-------|
| 0–2 | New Customer |
| 3–5 | Developing Customer |
| 6–8 | Established Customer |
| 9–10 | Loyal Customer |

In [15]:
# Create Tenure Groups

tenure_bins = [-1, 2, 5, 8, 10]

tenure_labels = [
    "New Customer",
    "Developing Customer",
    "Established Customer",
    "Loyal Customer"
]

clean_df["TenureGroup"] = pd.cut(
    clean_df["Tenure"],
    bins=tenure_bins,
    labels=tenure_labels
)

clean_df[["Tenure", "TenureGroup"]].sample(5, random_state=42)

,Tenure,TenureGroup
6252,3,Developing Customer
4684,1,New Customer
1731,4,Developing Customer
4742,8,Established Customer
4521,7,Established Customer


In [16]:
tenure_group_distribution = (
    clean_df["TenureGroup"]
    .value_counts()
    .sort_index()
    .reset_index(name="Customer Count")
)

tenure_group_distribution["Percentage (%)"] = (
    tenure_group_distribution["Customer Count"] / len(clean_df) * 100
).round(2)

tenure_group_distribution

,TenureGroup,Customer Count,Percentage (%)
0,New Customer,2496,24.96
1,Developing Customer,3010,30.10
2,Established Customer,3020,30.20
3,Loyal Customer,1474,14.74


### Observation

The **TenureGroup** feature was successfully created by classifying customers into four loyalty segments based on the length of their relationship with the bank.

The largest customer segment is **Established Customer** (**30.20%**), closely followed by **Developing Customer** (**30.10%**). **New Customers** account for **24.96%** of the dataset, while **Loyal Customers**, with the longest banking relationships, represent **14.74%**.

These tenure groups provide a meaningful way to analyze customer loyalty and will support future investigations into how the duration of a customer's relationship with the bank influences churn behaviour.

---

## D. Verifying Derived Segmentation Fields

Before proceeding to exploratory data analysis, the newly created segmentation fields are verified to ensure they have been added successfully and contain the expected values.

In [17]:
derived_columns = [
    "AgeGroup",
    "CreditScoreBand",
    "BalanceCategory",
    "TenureGroup"
]

clean_df[derived_columns].head()

,AgeGroup,CreditScoreBand,BalanceCategory,TenureGroup
0,30–45,Fair,Zero Balance,New Customer
1,30–45,Fair,Medium Balance,New Customer
2,30–45,Poor,High Balance,Established Customer
3,30–45,Good,Zero Balance,New Customer
4,30–45,Excellent,High Balance,New Customer


---

## Saving the Cleaned Dataset

The cleaned dataset, including the newly created segmentation fields, is saved for use in the subsequent notebooks. This ensures that future analyses begin with a consistent and prepared dataset.

In [19]:
clean_df.to_csv(
    "../data/processed/European_bank_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


# Conclusion

The data cleaning and preparation process was completed successfully, resulting in a structured dataset ready for exploratory analysis.

### Key Outcomes

- Removed non-analytical fields (`Surname`, `CustomerId`, and `Year`) to simplify the dataset.
- Converted categorical variables (`Geography` and `Gender`) to the `category` data type for improved memory efficiency and easier grouping.
- Created four derived segmentation fields:
  - **AgeGroup**
  - **CreditScoreBand**
  - **BalanceCategory**
  - **TenureGroup**
- Verified that all newly created features were generated successfully.
- Saved the cleaned dataset for use in subsequent notebooks.

With the dataset now cleaned and enriched, it is ready for **Notebook 03: Exploratory Data Analysis (EDA)**, where customer characteristics, churn patterns, and relationships between variables will be explored through descriptive statistics and visualizations.